In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# --- PATCH FP16 ---
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Injetar em RoTHP
try:
    import easy_tpp.model.torch_model.torch_rothp
    easy_tpp.model.torch_model.torch_rothp.attention = attention_fixed
except: pass

from easy_tpp.model.torch_model.torch_rothp import RoTHP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")


In [ ]:
def enable_dropout(m):
    """Ativa apenas as camadas de Dropout durante a inferência."""
    for each_module in m.modules():
        if isinstance(each_module, nn.Dropout):
            each_module.train()

def predict_mc_dropout(model, batch, n_samples=50):
    """
    Realiza N predições estocásticas mantendo Dropout ligado.
    Retorna média e desvio padrão das intensidades.
    """
    # 1. Colocar modelo em eval (congela BatchNorm), mas forçar Dropout on
    model.eval()
    enable_dropout(model)
    return None 

class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.2 # Aumentado para 0.2 para tornar a incerteza visível
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list, pad_id, time_scale):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / time_scale
        td = td / time_scale
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)


In [ ]:
# Carregar Dados
print("Loading Retweet...")
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

all_deltas = []
for item in train_data:
    td = item['time_since_last_event']
    all_deltas.extend([d for d in td if d > 0])
time_scale = np.mean(all_deltas)

num_types = 3
pad_id = 3
collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate, num_workers=0)

def train_rothp(epochs=30):
    print(f"\n>>> Treinando RoTHP (Dropout=0.2)...")
    config = ModelConfig(num_types, pad_id)
    model = RoTHP(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    for epoch in range(1, epochs+1):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f"  Ep {epoch}: Loss {total_loss:.4f}")
    return model

model = train_rothp()


In [ ]:
def visualize_uncertainty(model, sample_idx=0, n_samples=50):
    print(f"\nGerando Incerteza (MC Dropout N={n_samples}) para amostra {sample_idx}...")
    sample = [test_data[sample_idx]]
    # CORREÇÃO: remover [] pois sample já é uma lista
    batch = collate(sample)
    pad_time, pad_delta, pad_type, _, attn = [t.to(device) for t in batch]
    
    t_seq = pad_time[0].cpu().numpy()
    valid_len = (pad_type[0] != pad_id).sum().item()
    t_seq = t_seq[:valid_len]
    target_type = pad_type[0, 1].item()
    
    t_start_idx = 1
    t_end_idx = min(6, valid_len-1)
    
    # Ativar MC Dropout
    model.eval()
    enable_dropout(model)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for i in range(t_start_idx, t_end_idx):
        t_prev = t_seq[i]
        t_next = t_seq[i+1]
        dt_real = t_next - t_prev
        dts_scan = torch.linspace(0, dt_real, 50, device=device).view(1, 1, -1)
        
        intensities_samples = []
        
        with torch.no_grad():
            L = pad_time.shape[1]
            sample_dtimes = torch.zeros(1, L, 50, device=device)
            sample_dtimes[:, i, :] = dts_scan.squeeze()
            
            for _ in range(n_samples):
                lambdas = model.compute_intensities_at_sample_times(
                    pad_time, pad_delta, pad_type, sample_dtimes, attention_mask=attn
                )
                l_curve = lambdas[0, i, :, target_type].cpu().numpy()
                intensities_samples.append(l_curve)
        
        intensities_samples = np.array(intensities_samples)
        mu = intensities_samples.mean(axis=0)
        std = intensities_samples.std(axis=0)
        
        t_curve = t_prev + dts_scan.cpu().numpy().flatten()
        
        ax.plot(t_curve, mu, color='blue', linewidth=2, label='Mean Intensity' if i==t_start_idx else "")
        ax.fill_between(t_curve, mu - 2*std, mu + 2*std, color='blue', alpha=0.2, label='Uncertainty (2σ)' if i==t_start_idx else "")

    for t in t_seq[t_start_idx:t_end_idx+1]:
        ax.axvline(x=t, color='black', alpha=0.5, linestyle=':')
    
    ax.set_title(f'RoTHP Intensity with Uncertainty (MC Dropout) - Type {target_type}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Intensity')
    ax.legend()
    plt.show()

visualize_uncertainty(model, sample_idx=42)
visualize_uncertainty(model, sample_idx=100)
